# Chaos in continuous systems — exercise lab

This laboratory accompanies [the chapter](https://mvreeuwijk.github.io/chaosbook/python/cont3d.html). It runs
entirely in your browser (Python via Pyodide) — nothing to install, and any
changes you make are yours alone: re-opening the lab from the chapter resets it.

The full text of the exercises is in the
[chapter's exercise section](https://mvreeuwijk.github.io/chaosbook/python/cont3d.html#exercises); this notebook
gives you a running start on each one. The package `chaosbook` supplies
generic tools that work on *any* right-hand side `f(t, x, ...)`: Poincaré
sections (`cb.poincare_section`), stroboscopic sections
(`cb.stroboscopic_section`), maxima (`cb.maxima`) and Lyapunov exponents
(`cb.effective_lyapunov`, `cb.lyapunov_exponent`).

In [ ]:
# Run this cell first: it installs the book's package into the
# in-browser Python (takes a few seconds, needs to run once per visit).
%pip install -q chaosbook ipywidgets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp
from ipywidgets import interact, FloatSlider
import chaosbook as cb

## The Rössler system at full parameter freedom

The chapter's interactive figure steps through a precomputed grid of $c$
values. Here every slider move integrates the Rössler system afresh and shows
the orbit, the sequence of $x$-maxima and the return map $x_{n+1}$ against
$x_n$. Try $c=3$, $3.4$, $3.55$ and $4$: period 1, 2, 4 and chaos.

In [ ]:
def rossler_explorer(c=4.0, a=0.432, b=2.0, tmax=400):
    t = np.arange(0, tmax, 0.02)
    sol = solve_ivp(cb.rossler, [0, tmax], [1, 0, 0], t_eval=t, args=(a, b, c),
                    rtol=1e-8, atol=1e-10, method="DOP853")
    h = len(t) // 2                                   # discard the transient
    x, y, z = sol.y[:, h:]
    _, xm = cb.maxima(t[h:], x)
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(11, 3.4))
    ax1.plot(x, y, "b", linewidth=0.4)
    ax1.set_xlabel("$x$")
    ax1.set_ylabel("$y$")
    ax2.plot(xm, "kD", mfc="none", markersize=3)
    ax2.set_xlabel("$n$")
    ax2.set_ylabel("$x_n$")
    ax3.plot(xm[:-1], xm[1:], "kD", mfc="none", markersize=3)
    ax3.set_xlabel("$x_n$")
    ax3.set_ylabel("$x_{n+1}$")
    fig.tight_layout()
    plt.show()

interact(rossler_explorer,
         c=FloatSlider(4.0, min=2.0, max=6.0, step=0.05),
         a=FloatSlider(0.432, min=0.1, max=0.5, step=0.01),
         b=FloatSlider(2.0, min=0.2, max=3.0, step=0.1));

## Exercise: Poincaré section of the Rössler system

Full text: [in the chapter](https://mvreeuwijk.github.io/chaosbook/python/cont3d.html#exercises). The Rössler system with
$a=b=0.2$ and $c=5$, integrated with $dt=0.02$. `solve_ivp` with `t_eval`
gives the numerical data as arrays.

In [ ]:
a, b, c = 0.2, 0.2, 5.0
dt, N = 0.02, 15000
t = np.arange(N) * dt
sol = solve_ivp(cb.rossler, [0, t[-1]], [1, 0, 0], t_eval=t, args=(a, b, c),
                rtol=1e-8, atol=1e-10, method="DOP853")
x, y, z = sol.y
n0 = N // 5                                           # remove the transient

# a) crossings of y = 0 from positive to negative, found by looping over the data
xc, zc = [], []
for n in range(n0, N):
    if y[n - 1] > 0 and y[n] <= 0:
        xc.append(x[n])
        zc.append(z[n])
plt.plot(xc, zc, "k.")
plt.xlabel("$x$")
plt.ylabel("$z$")
plt.show()

In [ ]:
# b) the section on top of the attractor, and the package's exact version:
# cb.poincare_section locates each crossing by root finding.
tp, Xp = cb.poincare_section(cb.rossler, [1, 0, 0], t[-1], (1, 0.0), direction=-1,
                             discard=t[n0], a=a, b=b, c=c)
fig = plt.figure(figsize=(5, 4.5))
ax = fig.add_subplot(projection="3d")
ax.plot(x[n0:], y[n0:], z[n0:], "b", linewidth=0.3)
ax.plot(Xp[:, 0], Xp[:, 1], Xp[:, 2], "r.", markersize=3)
ax.set_xlabel("$x$"); ax.set_ylabel("$y$"); ax.set_zlabel("$z$")
plt.show()

# c) your turn: repeat for other values of c between 2 and 6.

## Exercise: Earth magnetic field

Full text: [parts a)–f) in the chapter](https://mvreeuwijk.github.io/chaosbook/python/cont3d.html#exercises). SymPy handles the
divergence, the fixed points and the Jacobian.

In [ ]:
x, y, z, b, k = sp.symbols("x y z b k", positive=True)
F = sp.Matrix([-sp.Rational(3, 4) * x + z * y, -b * y + (z - 1) * x, 1 - x * y])

# a) the divergence
print("divergence:", sp.simplify(sum(F[i].diff(v) for i, v in enumerate((x, y, z)))))

# b) b(k) and the proposed fixed points: all three components must vanish
bk = -sp.Rational(1, 3) + (-3 + 2 * k**2) ** 2 / (12 * k**4)
fp1 = {x: 1 / k, y: k, z: sp.Rational(3, 4) / k**2}
print([sp.simplify(f.subs(b, bk).subs(fp1)) for f in F])

# c) stability at k = 0.72 and k = 0.75: eigenvalues of the Jacobian
J = F.jacobian([x, y, z])
for kv in (0.72, 0.75):
    Jn = np.array(J.subs(b, bk).subs(fp1).subs(k, kv).evalf(), float)
    print("k =", kv, " eigenvalues:", np.round(np.linalg.eigvals(Jn), 4))

In [ ]:
# e) k = 0.72: integrate from a neighbour of a fixed point and plot t = 500...1000
kv, eps = 0.72, 0.5
bv = float(bk.subs(k, kv))
def earth(t, s, b=bv):
    x, y, z = s
    return [-0.75 * x + z * y, -b * y + (z - 1) * x, 1 - x * y]

fp = np.array([-1 / kv, -kv, 0.75 / kv**2])           # the other fixed point
ic = fp + eps
t = np.linspace(500, 1000, 10000)
sol = solve_ivp(earth, [0, 1000], ic, t_eval=t, rtol=1e-8, atol=1e-10)
fig = plt.figure(figsize=(5, 4.5))
ax = fig.add_subplot(projection="3d")
ax.plot(*sol.y, "r", linewidth=0.4)
ax.set_xlabel("$x$"); ax.set_ylabel("$y$"); ax.set_zlabel("$z$")
plt.show()
# Try eps = 1.0 as well, and discuss the result against your stability analysis.

## Exercise: Belousov-Zhabotinski reaction

Full text: [parts a)–e) in the chapter](https://mvreeuwijk.github.io/chaosbook/python/cont3d.html#exercises). The
Gaspard-Nicolis equations with the control parameter $d$; the system is
stiff ($\epsilon=0.01$), so use an implicit solver.

In [ ]:
a, b, c, f, g, l, s, eps = 0.5, 3.0, 5.0, 0.5, 0.6, 1.3, 0.3, 0.01

def bz(t, X, d=0.5):
    x, y, z = X
    return [(z - a * x**3 + b * x**2 - c * x) / eps,
            y * (z + s * x - l),
            z * (d * z - f * y - x + g)]

# a), b): time series and phase space for one value of d - vary d yourself
d, tend = 0.5, 200
t = np.linspace(tend / 2, tend, 5000)
sol = solve_ivp(bz, [0, tend], [1, 1, 1], t_eval=t, args=(d,), method="Radau",
                rtol=1e-8, atol=1e-10)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.2))
ax1.plot(t, sol.y[1], "k", linewidth=0.7)
ax1.set_xlabel("$t$")
ax1.set_ylabel("$y$")
ax2.plot(sol.y[0], sol.y[1], "b", linewidth=0.4)
ax2.set_xlabel("$x$")
ax2.set_ylabel("$y$")
plt.show()

In [ ]:
# c), d): fixed points as a function of d. With y != 0 and z != 0 the second
# equation gives z = l - s x and the third y = (d z - x + g)/f, so only a
# cubic in x remains - and it does not contain d.
x, y, z, d = sp.symbols("x y z d")
cubic = sp.expand(((l - s * x) - a * x**3 + b * x**2 - c * x))
xs = [r.real for r in np.roots([float(cu) for cu in sp.Poly(cubic, x).all_coeffs()])
      if abs(r.imag) < 1e-12]
print("real roots x* =", xs)
x1 = xs[0]
z1 = l - s * x1
y1 = sp.expand((d * z1 - x1 + g) / f)               # the d-dependent fixed point
print("fixed point:", x1, y1, z1)

# classify it for d = 0.25 and d = 0.35 with the eigenvalues of the Jacobian
Fs = sp.Matrix([(z - a * x**3 + b * x**2 - c * x) / eps,
                y * (z + s * x - l),
                z * (d * z - f * y - x + g)])
J = Fs.jacobian([x, y, z])
for dv in (0.25, 0.35):
    Jn = np.array(J.subs({x: x1, y: y1.subs(d, dv), z: z1, d: dv}).evalf(), float)
    print("d =", dv, " eigenvalues:", np.round(np.linalg.eigvals(Jn), 3))

# e) the divergence measures the 'dissipativeness'
print("divergence:", sp.simplify(sum(Fs[i].diff(v) for i, v in enumerate((x, y, z)))))

## Exercise: signal transmission in neurons

Full text: [parts a)–h) in the chapter](https://mvreeuwijk.github.io/chaosbook/python/cont3d.html#exercises). The
Bonhoeffer-van der Pol oscillator, written as the autonomous system with
$z=\omega t$. `cb.stroboscopic_section` samples it at $t_n = 2\pi n$ and
`cb.effective_lyapunov` gives the effective Lyapunov exponent.

In [ ]:
def bvp(t, X, A=0.0, B=0.25, omega=1.0):
    x, y, z = X
    return [x - x**3 / 3 - y + A + B * np.cos(z), x / 5 - 3 * y / 10, omega]

# e) A = 0, omega = 1, B = 0.25: the (x, y) projection from x0 = 0, y0 = 1, z0 = 0
sol = solve_ivp(bvp, [0, 400], [0, 1, 0], t_eval=np.linspace(200, 400, 5000),
                args=(0.0, 0.25, 1.0), rtol=1e-9, atol=1e-12, method="DOP853")
plt.plot(sol.y[0], sol.y[1], "b", linewidth=0.5)
plt.xlabel("$x$")
plt.ylabel("$y$")
plt.show()

In [ ]:
# f) stroboscopic section at t_n = 2 pi n for one value of B - scan B yourself
B = 0.38
P = cb.stroboscopic_section(bvp, [0, 1, 0], 2 * np.pi, 600, discard=100,
                            A=0.0, B=B, omega=1.0)
plt.plot(P[:, 0], P[:, 1], "k.", markersize=3)
plt.xlabel("$x_n$")
plt.ylabel("$y_n$")
plt.show()

In [ ]:
# g) effective Lyapunov exponent for B = 0.38: two trajectories 1e-8 apart
eps = 1e-8
t = np.arange(0, 100, 0.01)
kw = dict(t_eval=t, args=(0.0, 0.38, 1.0), rtol=1e-10, atol=1e-12, method="DOP853")
X1 = solve_ivp(bvp, [0, t[-1]], [0, 1, 0], **kw).y.T
X2 = solve_ivp(bvp, [0, t[-1]], [eps, 1, 0], **kw).y.T
le, local_rates = cb.effective_lyapunov(bvp, t, X1, X2, A=0.0, B=0.38, omega=1.0)
good = np.linalg.norm(X2 - X1, axis=1) < 1e-2        # perturbation still small
print("Lambda_e (time average):", le[good].mean())
# h) compare with the local stretching rates (eigenvalues of the symmetric part of
#    the Jacobian), local_rates[:, k]; their time averages are not Lyapunov exponents,
#    cb.lyapunov_spectrum_flow computes those

## Optional: your own orbit in the restricted three-body problem

The co-rotating equations, with the heavy bodies at $(-\mu, 0)$ and
$(1-\mu, 0)$, are `cb.rotating_threebody`; `cb.lagrange_points` and
`cb.threebody_potential` give the Lagrange points and the Hill region
$V>H_0$. One word of caution: never release the satellite at rest
($u_0=v_0=0$) with a realistic moon mass — it falls straight into a body and
the integration blows up. Always give it some speed, e.g. $v_0 > 0$.

In [ ]:
mu = 0.2
x0, y0, u0, v0 = 0.45, 0.1, 0.1, 0.4                  # v0 > 0: not released at rest
V = lambda x, y: cb.threebody_potential(x, y, mu)
xL1, xL2, xL3 = cb.lagrange_points(mu)
VL1, VL2 = V(xL1, 0), V(xL2, 0)
H0 = 0.5 * (u0**2 + v0**2) + V(x0, y0)
print("H0 =", H0, "  V_L1 =", VL1, "  V_L2 =", VL2)
if H0 < VL2:
    print("H0 < V_L2: confined to the neighbourhood of one body")
elif H0 < VL1:
    print("V_L2 < H0 < V_L1: can pass between the bodies but cannot escape")
else:
    print("H0 > V_L1: can escape")

t = np.linspace(0, 100, 10000)
sol = solve_ivp(cb.rotating_threebody, [0, 100], [x0, y0, u0, v0], t_eval=t, args=(mu,),
                rtol=1e-10, atol=1e-12, method="DOP853")
gx, gy = np.meshgrid(np.linspace(-1.5, 1.5, 300), np.linspace(-1.5, 1.5, 300))
plt.contourf(gx, gy, V(gx, gy), levels=[H0, 1e3], colors="0.85")   # forbidden: V > H0
plt.plot(sol.y[0], sol.y[1], "k", linewidth=0.4)
plt.plot([-mu, 1 - mu], [0, 0], "ko")
plt.xlim(-1.5, 1.5); plt.ylim(-1.5, 1.5); plt.gca().set_aspect("equal")
plt.xlabel("$x$"); plt.ylabel("$y$")
plt.show()
